# Name That LLM: TF-IDF + stylometry LinearSVC with a style-matched Open Pool sample + 2 two-stage LoRA classifiers

Generated by `name_that_llm/kaggle_notebook.py`; do not edit by hand.

* **Model:** character 1-5-grams + word 1-2-grams + 16 punctuation/style rates, then LinearSVC.
* **Data:** `train.csv` plus 2,000 passages per model from the public [Open Pool](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool), sampled by its `match_weight` so typography and length match the real passages.
* **LoRA members:** checkpoints from the training notebooks, blended by adding log-probabilities with weights `lora_out_4b_full` 1.0, `lora_out_8b_full` 2.0.
* **Post-processing:** partial Sinkhorn class balancing over the test set.
* **Local grouped CV** (5 folds by `prompt_id`): balanced accuracy 0.451 for the SVM alone.

Needs GPU T4 and no internet; about 15 minutes for the SVM plus a few per LoRA.

In [ ]:
import gc
import glob
import json
import os
import re
import time

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    # find the inputs wherever Kaggle mounts them; list /kaggle/input if one is missing
    attached = sorted(glob.glob('/kaggle/input/*/*', recursive=False))
    print('attached inputs:', attached)
    comps = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)
             if os.path.exists(os.path.join(os.path.dirname(p), 'test.csv'))]
    assert comps, f'Attach the name-that-llm competition data; found {attached}'
    COMP = comps[0]
    pools = glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True)
    assert pools, f'Attach crtproduction/name-that-llm-open-pool; found {attached}'
    POOL = pools[0]
    OUT = 'submission.parquet'
else:  # running from this project's notebooks/ folder
    COMP = '../data/raw'
    POOL = '../data/external/open_pool/ntl_open_pool.parquet'
    OUT = '../data/processed/submission_from_notebook.parquet'

# settings of the committed local model (reports/pool2000_w1_cv_metrics.json)
N_PER_LABEL, POOL_SEED, POOL_WEIGHT = 2000, 0, 1.0
C, STYLE_WEIGHT = 0.5, 0.2
BALANCE_TEMPERATURE, BALANCE_STRENGTH = 1.0, 0.3
USE_LORA, LORA_WEIGHTS, T_SVM = True, {'lora_out_4b_full': 1.0, 'lora_out_8b_full': 2.0}, 2.0
print('competition data:', COMP)
print('open pool:', POOL)
# LoRA members: each training notebook's output folder (lora_out*) + its base model.
# Checkpoints saved before meta.json recorded the base model are Qwen3-4B-Base.
DEFAULT_BASE = {'model_type': 'qwen3', 'hidden_size': 2560, 'num_hidden_layers': 36}


def find_base(spec):
    for cfg in sorted(glob.glob('/kaggle/input/**/config.json', recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f'attach the base model {spec}')


if ON_KAGGLE:
    found = {os.path.basename(os.path.dirname(p)): os.path.dirname(p)
             for p in glob.glob('/kaggle/input/**/lora_classifier.safetensors', recursive=True)}
else:  # NTL_LORA_DIRS='name=path,...'; one base model (NTL_LORA_BASE) for all of them
    found = dict(kv.split('=', 1) for kv in os.environ.get(
        'NTL_LORA_DIRS', 'lora_out=../data/interim/lora4b').split(','))
missing = sorted(set(LORA_WEIGHTS) - set(found))
assert not missing, f'Attach the LoRA training notebook outputs {missing}; found {found}'
LORA_DIRS = {name: found[name] for name in LORA_WEIGHTS}
# a 5-fold member keeps its fold adapters in fold0..fold4 next to its full fit; average them
LORA_ADAPTERS = {name: [d] + sorted(glob.glob(os.path.join(d, 'fold[0-9]')))
                 for name, d in LORA_DIRS.items()}
for name, d in LORA_DIRS.items():
    print(f'LoRA member {name} (weight {LORA_WEIGHTS[name]}): {d}, '
          f'{len(LORA_ADAPTERS[name])} adapter(s)')


## Features (copied from `name_that_llm/features.py`)

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler

# name -> regex; counted per 100 words
PUNCT_PATTERNS = {
    "em_dash": "—",
    "en_dash": "–",
    "semicolon": ";",
    "colon": ":",
    "comma": ",",
    "question": r"\?",
    "exclaim": "!",
    "parens": r"\(",
    "curly_quote": "[‘’“”]",
    "straight_quote": "['\"]",
    "digits": r"\d",
}
_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
_WORD = re.compile(r"[a-z']+")


def style_features(texts) -> pd.DataFrame:
    """Dense stylometric features: punctuation rates, length and lexical diversity."""
    s = pd.Series(list(texts), dtype=object)
    words = s.str.split().str.len().clip(lower=1)
    out = {name: s.str.count(pat) / words * 100 for name, pat in PUNCT_PATTERNS.items()}
    out["has_em_dash"] = s.str.contains("—").astype(float)
    out["has_curly"] = s.str.contains("[‘’“”]").astype(float)
    out["log_words"] = np.log(words)
    sentences = s.map(lambda t: max(1, len([x for x in _SENTENCE_SPLIT.split(t.strip()) if x])))
    out["words_per_sentence"] = words / sentences
    tokens = s.str.lower().map(_WORD.findall)
    out["avg_word_len"] = tokens.map(lambda w: np.mean([len(x) for x in w]) if w else 0.0)
    out["type_token"] = tokens.map(lambda w: len(set(w)) / max(1, len(w)))
    return pd.DataFrame(out).fillna(0.0)


class StyleFeatures(BaseEstimator, TransformerMixin):
    """sklearn wrapper around :func:`style_features`."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return style_features(X).to_numpy(dtype=float)

    def get_feature_names_out(self, input_features=None):
        return np.array(style_features(["a b."]).columns, dtype=object)


def build_features(style_weight: float = 1.0) -> FeatureUnion:
    """Character n-grams + word n-grams + scaled stylometric features."""
    return FeatureUnion(
        [
            (
                "char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(1, 5),
                    lowercase=False,
                    sublinear_tf=True,
                    min_df=2,
                    max_features=300_000,
                    dtype=np.float32,
                ),
            ),
            (
                "word",
                TfidfVectorizer(
                    analyzer="word",
                    ngram_range=(1, 2),
                    lowercase=True,
                    token_pattern=r"(?u)\b\w[\w']*\b",
                    sublinear_tf=True,
                    min_df=2,
                    dtype=np.float32,
                ),
            ),
            ("style", make_pipeline(StyleFeatures(), StandardScaler())),
        ],
        transformer_weights={"char": 1.0, "word": 1.0, "style": style_weight},
    )


## Class balancing (copied from `name_that_llm/modeling/postprocess.py`)

In [ ]:
def softmax(scores: np.ndarray, temperature: float) -> np.ndarray:
    z = scores * temperature
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def sinkhorn(proba: np.ndarray, strength: float = 1.0, n_iter: int = 50) -> np.ndarray:
    """Geometric mix of ``proba`` and its column-balanced Sinkhorn projection."""
    if strength <= 0:
        return proba
    n, k = proba.shape
    q = proba + 1e-12
    for _ in range(n_iter):
        q = q / q.sum(axis=0, keepdims=True) * (n / k)  # each class gets n/k total mass
        q = q / q.sum(axis=1, keepdims=True)  # each row is a distribution
    logp = (1 - strength) * np.log(proba + 1e-12) + strength * np.log(q)
    p = np.exp(logp - logp.max(axis=1, keepdims=True))
    return p / p.sum(axis=1, keepdims=True)


## Data: train.csv + a style-matched Open Pool sample

In [ ]:
train = pd.read_csv(f'{COMP}/train.csv')
test = pd.read_csv(f'{COMP}/test.csv')
pool = pd.read_parquet(POOL, columns=['text', 'model', 'match_weight'])

# same draw as name_that_llm/dataset.py: weighted sampling without replacement
# (Efraimidis-Spirakis) by match_weight, the same number of passages per label
parts = []
for label, d in pool.groupby('model'):
    rng = np.random.default_rng([POOL_SEED, len(parts)])
    keys = rng.random(len(d)) ** (1.0 / d.match_weight.clip(lower=1e-6).to_numpy())
    parts.append(d.iloc[np.argsort(-keys)[:min(N_PER_LABEL, len(d))]])
pool_sample = pd.concat(parts, ignore_index=True)
del pool

LABELS = sorted(train.model.unique())
assert len(LABELS) == 18 and set(pool_sample.model) <= set(LABELS)
print(f'train {len(train)}, pool sample {len(pool_sample)}, test {len(test)}')

## Train

In [ ]:
X = pd.concat([train.text, pool_sample.text], ignore_index=True)
y = np.concatenate([train.model.to_numpy(), pool_sample.model.to_numpy()])
w = np.concatenate([np.ones(len(train)), np.full(len(pool_sample), POOL_WEIGHT)])

model = Pipeline([
    ('features', build_features(STYLE_WEIGHT)),
    ('clf', LinearSVC(C=C, class_weight='balanced', max_iter=5000)),
])
t0 = time.time()
model.fit(X, y, clf__sample_weight=w)
print(f'trained in {time.time() - t0:.0f}s; in-sample balanced acc on train.csv: '
      f'{np.mean([np.mean(model.predict(train.text[train.model == m]) == m) for m in LABELS]):.3f}')

## SVM scores for the test set

In [ ]:
svm_scores = model.decision_function(test.text)
CLASSES = np.asarray(model.classes_)
del model, X, y, w, pool_sample
gc.collect()

## LoRA classifier (copied from `name_that_llm/lora.py`)

In [ ]:
import json
import math
import os
import time

import numpy as np
import torch
from torch import nn
from transformers import AutoConfig, AutoModel, AutoTokenizer

LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update ``B @ A * alpha / r``."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, device=dev))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        delta = self.drop(x).to(self.lora_A.dtype) @ self.lora_A.T @ self.lora_B.T
        return self.base(x) + (delta * self.scale).to(x.dtype)


def add_lora(model: nn.Module, r: int, alpha: float, dropout: float) -> int:
    """Wrap every target linear layer of ``model`` in LoRALinear; returns how many."""
    targets = [
        (parent, name)
        for parent in model.modules()
        for name, child in parent.named_children()
        if isinstance(child, nn.Linear) and name in LORA_TARGETS
    ]
    for parent, name in targets:
        setattr(parent, name, LoRALinear(getattr(parent, name), r, alpha, dropout))
    return len(targets)


def load_trunk(base_dir: str, keep: int, dtype, device) -> nn.Module:
    """The first ``keep`` decoder layers of the base model, frozen, final norm switched off."""
    cfg = AutoConfig.from_pretrained(base_dir)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, "layer_types", None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base_dir, config=cfg, dtype=dtype).to(device)
    trunk.norm = nn.Identity()
    trunk.requires_grad_(False)
    return trunk


class LoRAClassifier(nn.Module):
    def __init__(
        self, trunk: nn.Module, layers, n_classes: int, r=16, alpha=32, dropout=0.05, ckpt=True
    ):
        super().__init__()
        self.trunk, self.layers = trunk, list(layers)
        self.n_lora = add_lora(trunk, r, alpha, dropout)
        if ckpt:  # recompute activations in backward: far less memory for ~30% more time
            trunk.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        H = trunk.config.hidden_size
        dev = trunk.embed_tokens.weight.device
        self.norms = nn.ModuleList([nn.LayerNorm(H) for _ in self.layers]).to(dev)
        self.head = nn.Linear(H * len(self.layers), n_classes).to(dev)

    def forward(self, input_ids, attention_mask):
        hs = self.trunk(
            input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True
        ).hidden_states
        m = attention_mask.unsqueeze(-1).float()
        m[:, 0] = 0  # the first token is an attention sink; leave it out of the mean
        den = m.sum(1).clamp(min=1)
        with torch.autocast(device_type=input_ids.device.type, enabled=False):
            z = torch.cat(
                [nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(self.layers, self.norms)],
                -1,
            )
            return self.head(z)

    def trainable_state(self) -> dict:
        """Only what training changes: LoRA matrices, LayerNorms and the head."""
        return {k: v.detach().cpu() for k, v in self.named_parameters() if v.requires_grad}


def tokenize(tok, texts, max_len: int) -> list[list[int]]:
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=max_len - 1)
    return [x + [tok.eos_token_id] for x in ids["input_ids"]]


def batches(ids, bs: int, shuffle: bool, rng=None):
    """Index batches of similar length (less padding); batch order shuffled when training."""
    if not shuffle:
        order = np.argsort([len(x) for x in ids], kind="stable")
    else:  # sort within random chunks of 50 batches so batches still differ between epochs
        order = rng.permutation(len(ids))
        chunk = bs * 50
        order = np.concatenate(
            [
                sorted(order[i : i + chunk], key=lambda j: len(ids[j]))
                for i in range(0, len(order), chunk)
            ]
        )
    out = [order[i : i + bs] for i in range(0, len(order), bs)]
    if shuffle:
        out = [out[i] for i in rng.permutation(len(out))]
    return out


def collate(tok, ids, idx, device):
    enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt")
    return enc["input_ids"].to(device), enc["attention_mask"].to(device)


def train_stage(
    model,
    tok,
    texts,
    y,
    *,
    epochs: float,
    lr_lora: float,
    lr_head: float,
    bs: int,
    max_len: int,
    n_classes: int,
    amp_dtype,
    seed: int = 0,
    max_steps: int | None = None,
    log_every: int = 50,
    log=print,
):
    """One training stage; class-weighted CE, AdamW, 10% warm-up then cosine decay."""
    device = model.head.weight.device
    rng = np.random.default_rng(seed)
    ids = tokenize(tok, texts, max_len)
    y = torch.tensor(np.array(y), device=device)
    counts = torch.bincount(y, minlength=n_classes).float()
    class_w = (counts.sum() / (n_classes * counts.clamp(min=1))).to(device)

    per_epoch = math.ceil(len(ids) / bs)
    total = math.ceil(epochs * per_epoch)
    if max_steps is not None:
        total = min(total, max_steps)
    lora = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n]
    rest = [p for n, p in model.named_parameters() if p.requires_grad and "lora_" not in n]
    opt = torch.optim.AdamW(
        [{"params": lora, "lr": lr_lora}, {"params": rest, "lr": lr_head}], weight_decay=0.01
    )
    warm = max(1, int(0.1 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (
            (s + 1) / warm
            if s < warm
            else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
        ),
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and amp_dtype == torch.float16)

    model.train()
    step, skipped, run_loss, run_n, t0 = 0, 0, 0.0, 0, time.time()
    while step < total:
        for idx in batches(ids, bs, shuffle=True, rng=rng):
            if step >= total:
                break
            input_ids, mask = collate(tok, ids, idx, device)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                logits = model(input_ids, mask)
            loss = nn.functional.cross_entropy(logits.float(), y[idx], weight=class_w)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1  # fp16 overflow in the backbone; drop this batch
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(lora + rest, 1.0)
                scaler.step(opt)
                scaler.update()
                run_loss += loss.item()
                run_n += 1
            sched.step()
            step += 1
            if step % log_every == 0 or step == total:
                el = time.time() - t0
                log(
                    f"step {step}/{total} loss {run_loss / max(1, run_n):.4f} "
                    f"{el / step:.2f}s/step, ~{el / step * (total - step) / 60:.0f} min left, "
                    f"skipped {skipped}"
                )
                run_loss, run_n = 0.0, 0
    model.eval()
    return {"steps": total, "skipped": skipped, "seconds": time.time() - t0}


@torch.no_grad()
def predict_logits(model, tok, texts, *, bs: int, max_len: int, amp_dtype) -> np.ndarray:
    device = model.head.weight.device
    ids = tokenize(tok, texts, max_len)
    out = np.zeros((len(ids), model.head.out_features), np.float32)
    model.eval()
    for idx in batches(ids, bs, shuffle=False):
        input_ids, mask = collate(tok, ids, idx, device)
        with torch.autocast(
            device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"
        ):
            out[idx] = model(input_ids, mask).float().cpu().numpy()
    return out


def load_tokenizer(base_dir: str):
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok


def save_checkpoint(model, out_dir: str, meta: dict):
    from safetensors.torch import save_file

    os.makedirs(out_dir, exist_ok=True)
    save_file(
        {k: v.contiguous() for k, v in model.trainable_state().items()},
        os.path.join(out_dir, "lora_classifier.safetensors"),
    )
    with open(os.path.join(out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)


def load_checkpoint(model, ckpt_dir: str):
    from safetensors.torch import load_file

    state = load_file(os.path.join(ckpt_dir, "lora_classifier.safetensors"))
    missing = model.load_state_dict(state, strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys
    return model


## LoRA logits for the test set, one member at a time

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    amp = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
else:
    device, amp = torch.device('cpu'), torch.float32
def log_softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))


lora_logits = {}  # per member: the mean log-probabilities of its adapters
for name, lora_dir in LORA_DIRS.items():
    meta = json.load(open(f'{lora_dir}/meta.json'))
    assert meta['labels'] == list(CLASSES), f'{name}: label order differs from the SVM'
    cfg = meta['cfg']
    base = find_base(cfg.get('base', DEFAULT_BASE)) if ON_KAGGLE else os.environ.get('NTL_LORA_BASE', '../data/interim/tiny_qwen3')
    t0 = time.time()
    tok = load_tokenizer(base)
    trunk = load_trunk(base, cfg['keep'], amp, device)
    lora_model = LoRAClassifier(trunk, cfg['layers'], len(CLASSES), cfg['r'], cfg['alpha'], 0.0, ckpt=False)
    logps = []
    for adapter in LORA_ADAPTERS[name]:
        assert json.load(open(f'{adapter}/meta.json'))['labels'] == list(CLASSES)
        load_checkpoint(lora_model, adapter)
        z = predict_logits(lora_model, tok, test.text, bs=16, max_len=cfg['max_len'], amp_dtype=amp)
        assert np.isfinite(z).all(), f'{adapter}: non-finite LoRA logits'
        logps.append(log_softmax(z))
    lora_logits[name] = np.mean(logps, axis=0)
    print(f'{name}: {len(logps)} adapter(s) in {time.time() - t0:.0f}s on {device} from {base}; '
          f"held-out balanced accuracy at training: fold 0 {meta.get('holdout_balanced_accuracy', float('nan')):.4f}, "
          f"all folds {meta.get('oof_balanced_accuracy', float('nan')):.4f}")
    del lora_model, trunk  # free the GPU for the next member
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

## Blend, balance, write `submission.parquet`

In [ ]:
def log_softmax(z):
    z = z - z.max(axis=1, keepdims=True)
    return z - np.log(np.exp(z).sum(axis=1, keepdims=True))


if USE_LORA:
    logp = log_softmax(svm_scores * T_SVM)
    for name, w_lora in LORA_WEIGHTS.items():
        logp = logp + w_lora * log_softmax(lora_logits[name])
    proba = softmax(logp, 1.0)
else:
    proba = softmax(svm_scores, BALANCE_TEMPERATURE)
pred = CLASSES[sinkhorn(proba, BALANCE_STRENGTH).argmax(axis=1)]
submission = pd.DataFrame({'id': test.id, 'model': pred})

assert submission.id.is_unique and len(submission) == len(test)
assert submission.model.notna().all() and submission.model.isin(LABELS).all()
submission.to_parquet(OUT, index=False)
print(submission.model.value_counts().to_string())
submission.head()